In [3]:
import ROOT
import math
import os

%jsroot on

# ============================================================
# SETTINGS
# ============================================================

file_path = "/root/geant4/detector/Tumor/Tumor1/tumor1.root"
tree_name = "t"

selected_volume = 3
selected_process = 2013
selected_pdg = 22

# Incident beam direction:
# /gps/direction 0 -1 0
incident_direction = (0.0, -1.0, 0.0)

# Histogram binning
n_theta_bins = 90       # 2-degree bins
theta_min = 0.0
theta_max = 180.0

n_energy_bins = 140     # 5-keV bins
energy_min = 0.0
energy_max = 700.0


# ============================================================
# OPEN ROOT FILE
# ============================================================

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"ROOT file not found:\n{file_path}"
    )

f = ROOT.TFile.Open(file_path)

if not f or f.IsZombie():
    raise RuntimeError(
        f"Could not open ROOT file:\n{file_path}"
    )

t = f.Get(tree_name)

if not t:
    f.ls()
    raise RuntimeError(
        f"Tree '{tree_name}' was not found."
    )

print("Tree entries:", t.GetEntries())


# ============================================================
# REMOVE OLD ROOT OBJECTS
# ============================================================

for object_name in [
    "h_theta_counts",
    "h_theta_et"
]:
    old_object = ROOT.gROOT.FindObject(object_name)

    if old_object:
        old_object.Delete()


for canvas_name in [
    "c_all",
    "c_theta_counts",
    "c_theta_et",
    "c_theta_et_3d"
]:
    old_canvas = ROOT.gROOT.FindObject(canvas_name)

    if old_canvas:
        old_canvas.Close()


# ============================================================
# CREATE HISTOGRAMS
# ============================================================

# Figure 1: theta vs counts
h_theta_counts = ROOT.TH1F(
    "h_theta_counts",
    "Compton scattering-angle distribution, volume 3;"
    "Scattering angle #theta (degrees);"
    "Counts per angular bin",
    n_theta_bins,
    theta_min,
    theta_max
)

h_theta_counts.SetStats(0)
h_theta_counts.SetLineWidth(2)


# Figures 2 and 3:
# x = theta
# y = et
# bin content = counts
h_theta_et = ROOT.TH2F(
    "h_theta_et",
    "Compton energy-angle distribution, volume 3;"
    "Scattering angle #theta (degrees);"
    "et (keV);"
    "Counts per bin",
    n_theta_bins,
    theta_min,
    theta_max,
    n_energy_bins,
    energy_min,
    energy_max
)

h_theta_et.SetStats(0)


# ============================================================
# NORMALIZE INCIDENT DIRECTION
# ============================================================

ix, iy, iz = incident_direction

incident_norm = math.sqrt(
    ix**2 + iy**2 + iz**2
)

if incident_norm <= 0:
    raise ValueError(
        "Incident direction cannot be zero."
    )

ix /= incident_norm
iy /= incident_norm
iz /= incident_norm


# ============================================================
# EVENT LOOP
# ============================================================

all_matching_records = 0
selected_points = 0
zero_momentum_records = 0
invalid_et_records = 0

for event in t:

    n = min(
        len(event.pdg),
        len(event.pro),
        len(event.vlm),
        len(event.et),
        len(event.px),
        len(event.py),
        len(event.pz)
    )

    for i in range(n):

        # Selection
        if int(event.vlm[i]) != selected_volume:
            continue

        if int(event.pro[i]) != selected_process:
            continue

        if int(event.pdg[i]) != selected_pdg:
            continue

        all_matching_records += 1

        # Momentum components
        px = float(event.px[i])
        py = float(event.py[i])
        pz = float(event.pz[i])

        p = math.sqrt(
            px**2 + py**2 + pz**2
        )

        if p <= 0:
            zero_momentum_records += 1
            continue

        # Outgoing photon unit vector
        ux = px / p
        uy = py / p
        uz = pz / p

        # Scattering angle from dot product
        cos_theta = (
            ix * ux +
            iy * uy +
            iz * uz
        )

        cos_theta = max(
            -1.0,
            min(1.0, cos_theta)
        )

        theta = math.degrees(
            math.acos(cos_theta)
        )

        # Energy quantity
        et_value = float(event.et[i])

        if not math.isfinite(et_value):
            invalid_et_records += 1
            continue

        # Fill histograms
        h_theta_counts.Fill(theta)
        h_theta_et.Fill(theta, et_value)

        selected_points += 1


# ============================================================
# PRINT SUMMARY
# ============================================================

print("\nSelection:")
print("vlm =", selected_volume)
print("pro =", selected_process)
print("pdg =", selected_pdg)

print("\nAll matching records:", all_matching_records)
print("Valid plotted points:", selected_points)
print("Zero-momentum records:", zero_momentum_records)
print("Invalid et records:", invalid_et_records)

print("\nTheta histogram entries:",
      h_theta_counts.GetEntries())

print("Theta-et histogram entries:",
      h_theta_et.GetEntries())


# ============================================================
# FIND HIGHEST THETA-COUNT BIN
# ============================================================

max_theta_bin = h_theta_counts.GetMaximumBin()

theta_peak = (
    h_theta_counts
    .GetXaxis()
    .GetBinCenter(max_theta_bin)
)

theta_peak_counts = (
    h_theta_counts
    .GetBinContent(max_theta_bin)
)

print("\nHighest theta-count bin:")
print(f"theta = {theta_peak:.2f} degrees")
print(f"counts = {theta_peak_counts:.0f}")


# ============================================================
# FIND HIGHEST THETA-ET-COUNT BIN
# ============================================================

max_counts = -1.0
max_bx = -1
max_by = -1

for bx in range(
    1,
    h_theta_et.GetNbinsX() + 1
):
    for by in range(
        1,
        h_theta_et.GetNbinsY() + 1
    ):

        counts = h_theta_et.GetBinContent(
            bx,
            by
        )

        if counts > max_counts:
            max_counts = counts
            max_bx = bx
            max_by = by

theta_et_peak = (
    h_theta_et
    .GetXaxis()
    .GetBinCenter(max_bx)
)

et_peak = (
    h_theta_et
    .GetYaxis()
    .GetBinCenter(max_by)
)

print("\nHighest theta-et bin:")
print(
    f"(theta, et, counts) = "
    f"({theta_et_peak:.2f} degrees, "
    f"{et_peak:.2f} keV, "
    f"{max_counts:.0f})"
)


# ============================================================
# CREATE ONE CANVAS WITH THREE PANELS
# ============================================================

c_all = ROOT.TCanvas(
    "c_all",
    "Compton plots",
    1500,
    1100
)

c_all.Divide(2, 2)


# ============================================================
# PANEL 1: THETA VS COUNTS
# ============================================================

pad1 = c_all.cd(1)

pad1.SetLeftMargin(0.12)
pad1.SetRightMargin(0.05)
pad1.SetBottomMargin(0.12)
pad1.SetTopMargin(0.08)

pad1.SetLogy(0)   # change to 1 for log Y

h_theta_counts.SetMinimum(0.0)
h_theta_counts.Draw("HIST")


# ============================================================
# PANEL 2: THETA VS ET
# COLOR = COUNTS
# ============================================================

pad2 = c_all.cd(2)

pad2.SetLeftMargin(0.11)
pad2.SetRightMargin(0.15)
pad2.SetBottomMargin(0.12)
pad2.SetTopMargin(0.08)

pad2.SetLogz(0)   # change to 1 for log color scale

h_theta_et.Draw("COLZ")


# ============================================================
# PANEL 3: 3D THETA VS ET VS COUNTS
# ============================================================

pad3 = c_all.cd(3)

pad3.SetLeftMargin(0.10)
pad3.SetRightMargin(0.14)
pad3.SetBottomMargin(0.10)
pad3.SetTopMargin(0.08)

pad3.SetLogz(0)   # change to 1 for log counts

h_theta_et.GetXaxis().SetTitleOffset(1.5)
h_theta_et.GetYaxis().SetTitleOffset(1.7)
h_theta_et.GetZaxis().SetTitleOffset(1.2)

h_theta_et.Draw("LEGO2Z")

pad3.SetTheta(25)
pad3.SetPhi(35)


# ============================================================
# PANEL 4: OPTIONAL TEXT SUMMARY
# ============================================================

pad4 = c_all.cd(4)
pad4.SetFillStyle(0)

summary = ROOT.TPaveText(
    0.08,
    0.20,
    0.92,
    0.85,
    "NDC"
)

summary.SetFillStyle(0)
summary.SetBorderSize(1)
summary.SetTextAlign(12)
summary.SetTextSize(0.04)

summary.AddText("Selection")
summary.AddText(f"vlm = {selected_volume}")
summary.AddText(f"pro = {selected_process}")
summary.AddText(f"pdg = {selected_pdg}")
summary.AddText("")
summary.AddText(
    f"Valid points = {selected_points}"
)
summary.AddText(
    f"Peak angle = {theta_peak:.2f} deg"
)
summary.AddText(
    f"Peak angular counts = {theta_peak_counts:.0f}"
)
summary.AddText("")
summary.AddText(
    f"Peak 2D bin:"
)
summary.AddText(
    f"theta = {theta_et_peak:.2f} deg"
)
summary.AddText(
    f"et = {et_peak:.2f} keV"
)
summary.AddText(
    f"counts = {max_counts:.0f}"
)

summary.Draw()


# ============================================================
# DISPLAY
# ============================================================

c_all.Modified()
c_all.Update()
c_all.Draw()

Tree entries: 100000

Selection:
vlm = 3
pro = 2013
pdg = 22

All matching records: 8469
Valid plotted points: 8469
Zero-momentum records: 0
Invalid et records: 0

Theta histogram entries: 8469.0
Theta-et histogram entries: 8469.0

Highest theta-count bin:
theta = 39.00 degrees
counts = 222

Highest theta-et bin:
(theta, et, counts) = (25.00 degrees, 72.50 keV, 96)
